## Langfuse LLM Observability and Prompt Management Tutorial

### What is Langfuse?
Langfuse is a powerful platform for:

* LLM Observability: Track and monitor LLM interactions in production
* Tracing: Detailed execution traces for debugging and optimization
* Analytics: Performance metrics and cost tracking
* Evaluation: Automated quality assessment of LLM outputs

In [24]:
%pip install langfuse langchain langchain-google-genai -q

### Environment Configuration

In [25]:
import os
from google.colab import userdata

os.environ["LANGFUSE_SECRET_KEY"] = userdata.get('LANGFUSE_SECRET_KEY')
os.environ["LANGFUSE_PUBLIC_KEY"] = userdata.get('LANGFUSE_PUBLIC_KEY')
os.environ["LANGFUSE_HOST"] = "https://cloud.langfuse.com"

# Securely get Google API key from Colab secrets
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

Purpose: Set up authentication for both Langfuse and ChatGoogleGenerativeAI services.
⚠️ Security Note: In production, use environment variables or secure credential management instead of hardcoding API keys

### Initialize Langfuse Client

In [26]:
from langfuse import get_client
from langfuse.langchain import CallbackHandler

# Initialize Langfuse client (prompt management)
langfuse = get_client()

# Initialize Langfuse CallbackHandler for Langchain (tracing)
langfuse_callback_handler = CallbackHandler()

### Prompt Management - Create Prompt Template

In [27]:
langfuse.create_prompt(
    name="event-planner",
    prompt=
    "Plan an event titled {{Event Name}}. The event will be about: {{Event Description}}. "
    "The event will be held in {{Location}} on {{Date}}. "
    "Consider the following factors: audience, budget, venue, catering options, and entertainment. "
    "Provide a detailed plan including potential vendors and logistics.",
    config={
        "model":"gemini-2.0-flash",
        "temperature": 0,
    },
    labels=["production"]
);

### Retrieve and Use Prompt

In [28]:
# Get current production version of prompt
langfuse_prompt = langfuse.get_prompt("event-planner")

In [29]:
print(langfuse_prompt.prompt)

Plan an event titled {{Event Name}}. The event will be about: {{Event Description}}. The event will be held in {{Location}} on {{Date}}. Consider the following factors: audience, budget, venue, catering options, and entertainment. Provide a detailed plan including potential vendors and logistics.


In [30]:
from langchain_core.prompts import ChatPromptTemplate

langchain_prompt = ChatPromptTemplate.from_template(
        langfuse_prompt.get_langchain_prompt(),
        metadata={"langfuse_prompt": langfuse_prompt},
    )

### Model Configuration from Prompt

In [31]:
model = langfuse_prompt.config["model"]
temperature = str(langfuse_prompt.config["temperature"])
print(f"Prompt model configurations\nModel: {model}\nTemperature: {temperature}")

Prompt model configurations
Model: gemini-2.0-flash
Temperature: 0


In [32]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0.7
)

chain = langchain_prompt | model

### Execute with Tracing

In [33]:
example_input = {
    "Event Name": "Wedding",
    "Event Description": "The wedding of Julia and Alex, a charming couple who share a love for art and nature. This special day will celebrate their journey together with a blend of traditional and contemporary elements, reflecting their unique personalities.",
    "Location": "Central Park, New York City",
    "Date": "June 5, 2024"
}

In [34]:
# we pass the callback handler to the chain to trace the run in Langfuse
response = chain.invoke(input=example_input,config={"callbacks":[langfuse_callback_handler]})

print(response.content)

Okay, let's plan an event titled **{Event Name}** taking place in Central Park, New York City on June 5, 2024, with the theme: **{Event Description}**.

To make this plan as comprehensive as possible, I'll need you to fill in those bracketed sections with the specific details of your event.  In the meantime, I will provide a robust framework that you can customize with your particular information.

**Here's a detailed event plan template:**

**I. EVENT OVERVIEW**

*   **Event Name:** {Event Name}
*   **Date:** June 5, 2024
*   **Time:**  (Placeholder - Specify Start and End Times, e.g., 10:00 AM - 4:00 PM)
*   **Location:** Central Park, New York City (Specific Location to be determined - see Venue section below)
*   **Event Description:** {Event Description}
*   **Event Goal:** (Clearly define what you want to achieve with this event. E.g., Raise awareness for X, celebrate Y, fundraise for Z, etc.)

**II. TARGET AUDIENCE**

*   **Demographics:** (Specify age range, gender, interests, 